# NB13 — Passive Three-Channel AE Baseline

**Version:** 1 · **Author:** CB · **Date:** 2026-06-30
**Acquisition notebook — requires DS1104Z connected via VISA.**

---

## 1  Objective

Establish a passive (no forced excitation) acoustic-emission baseline for
the three-sensor configuration introduced on 2026-06-30:

| Channel | Sensor | Probe | Scale |
|---------|--------|-------|-------|
| CH1 | Soil piezo, bare in substrate, LM358 + 830 kΩ | 10× | 0.5 V/div |
| CH2 | **Defective** | — | — |
| CH3 | Rod piezo A, LM358 + 820 kΩ | 10× | 0.5 V/div |
| CH4 | Rod piezo B, LM358 + 820 kΩ | 10× | 0.5 V/div |

Metrics reported per channel:
- Noise floor: RMS voltage and peak-to-peak in the absence of known events
- Power spectral density (Welch, Hann window)
- Inter-channel coherence CH1–CH3 and CH1–CH4 (soil–rod coupling check)
- Time-domain event detection (threshold crossings)

This notebook is intentionally **separate** from NB00 (offline SSOT) so that
NB00 remains fully reproducible without hardware.

---

## 2  Configuration


In [ ]:
import sys, time, json, subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

from plant_ae.watering import CHARACTERIZATION_ROOT, CONFIG_PATH
CHAR_ROOT = CHARACTERIZATION_ROOT
if str(CHAR_ROOT) not in sys.path:
    sys.path.insert(0, str(CHAR_ROOT))

from scope.config import load_config
from scope.instrument import InstrumentConnection, acquire_waveform_bytes

# ── Acquisition parameters ────────────────────────────────────────────────────
SENSOR_CHANNELS = (1, 3, 4)
CH_PROBE  = {1: 10,   3: 10,   4: 10}
CH_SCALE  = {1: 0.5,  3: 0.5,  4: 0.5}    # V/div
N_FRAMES  = 60                              # ~60 captures at 1.2 ms each
TRIG_SRC  = 'CHAN4'
TRIG_LEVEL_V = 0.010                        # 10 mV

RUN_ID  = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = CHAR_ROOT / 'data' / 'passive_ae_baseline' / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run ID:  {RUN_ID}')
print(f'Output:  {OUT_DIR}')


## 3  Instrument Configuration

In [ ]:
def configure_passive(conn):
    conn.write(':STOP'); time.sleep(0.3)
    conn.write(':CHAN2:DISP OFF')                    # CH2 defective
    for ch in SENSOR_CHANNELS:
        conn.write(f':CHAN{ch}:DISP ON')
        conn.write(f':CHAN{ch}:COUP AC')
        conn.write(f':CHAN{ch}:PROB {CH_PROBE[ch]}')
        conn.write(f':CHAN{ch}:SCAL {CH_SCALE[ch]}')
    conn.write(f':TRIG:MODE EDGE')
    conn.write(f':TRIG:EDGE:SOUR {TRIG_SRC}')
    conn.write(f':TRIG:EDGE:SLOP POS')
    conn.write(f':TRIG:EDGE:LEV {TRIG_LEVEL_V}')
    conn.write(':TRIG:SWE NORM')
    conn.write(':ACQ:TYPE NORM')
    conn.write(':ACQ:MEMD NORM')
    conn.write(':RUN')
    time.sleep(0.5)
    print('Scope configured.')
    print(f'  Trigger: EDGE {TRIG_SRC}, level {TRIG_LEVEL_V*1000:.0f} mV, POS slope')
    for ch in SENSOR_CHANNELS:
        print(f'  CH{ch}: PROB={CH_PROBE[ch]}, SCAL={CH_SCALE[ch]} V/div, AC')


## 4  Acquisition

In [ ]:
records = []
skipped = 0

config = load_config(CONFIG_PATH)
print(f'Acquiring {N_FRAMES} frames...')

with InstrumentConnection(config) as conn:
    configure_passive(conn)
    for i in range(N_FRAMES):
        frame = {'frame': i, 'ts': datetime.now(timezone.utc).isoformat()}
        ok = True
        for ch in SENSOR_CHANNELS:
            try:
                raw = acquire_waveform_bytes(conn, ch)
                v   = raw['voltage']
                fs  = raw['sample_rate']
                frame[f'v{ch}'] = v.tolist()
                frame[f'fs{ch}'] = float(fs)
            except Exception as e:
                print(f'  [WARN] frame {i} CH{ch}: {e}')
                ok = False; break
        if ok:
            # Save scalar stats per frame
            stat = {'frame': i, 'ts': frame['ts']}
            for ch in SENSOR_CHANNELS:
                v = np.array(frame[f'v{ch}'])
                stat[f'rms{ch}_V']  = float(np.sqrt(np.mean(v**2)))
                stat[f'pp{ch}_V']   = float(v.max() - v.min())
                stat[f'fs{ch}_hz']  = frame[f'fs{ch}']
                stat[f'n{ch}']      = len(v)
            with open(OUT_DIR / 'records.jsonl', 'a') as f:
                f.write(json.dumps(stat) + '\n')
            # Save first 5 frames as waveforms
            if i < 5:
                np.savez_compressed(OUT_DIR / f'frame_{i:04d}.npz',
                                    **{f'ch{ch}': np.array(frame[f'v{ch}']) for ch in SENSOR_CHANNELS},
                                    fs=frame['fs1'])
            records.append(stat)
            if (i+1) % 10 == 0:
                rms_str = '  '.join(f'CH{ch}={stat[f"rms{ch}_V"]*1000:.3f}mV' for ch in SENSOR_CHANNELS)
                print(f'  [{i+1:3d}/{N_FRAMES}]  {rms_str}')
        else:
            skipped += 1

print(f'Done: {len(records)} frames acquired, {skipped} skipped')


## 5  Noise-Floor and Spectral Analysis

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.signal import welch, coherence

# Load all NPZ waveforms for spectral analysis
npz_files = sorted(OUT_DIR.glob('frame_*.npz'))
waveforms  = {ch: [] for ch in SENSOR_CHANNELS}
fs_val     = None

for fp in npz_files:
    d = np.load(fp)
    for ch in SENSOR_CHANNELS:
        waveforms[ch].append(d[f'ch{ch}'])
    fs_val = float(d['fs'])

# Also load all scalar records
if not records:
    records = [json.loads(l) for l in (OUT_DIR/'records.jsonl').read_text().splitlines() if l.strip()]

# ── Noise floor table ─────────────────────────────────────────────────────────
print('=== Noise Floor (all frames) ===')
print(f'  {"Channel":>10}  {"RMS [mV]":>10}  {"P-P [mV]":>10}')
for ch in SENSOR_CHANNELS:
    rms_vals = [r[f'rms{ch}_V']*1000 for r in records]
    pp_vals  = [r[f'pp{ch}_V' ]*1000 for r in records]
    print(f'  {"CH"+str(ch):>10}  {np.mean(rms_vals):>10.4f}  {np.mean(pp_vals):>10.3f}')

# ── PSD ───────────────────────────────────────────────────────────────────────
if not waveforms[1] and not waveforms[3]:
    print('\nNo waveform NPZ files found — run the acquisition cell first.')
else:
    CH_COLORS = {1: '#1f77b4', 3: '#2ca02c', 4: '#d62728'}
    CH_LABELS = {1: 'CH1 (Soil piezo)', 3: 'CH3 (Rod A)', 4: 'CH4 (Rod B)'}

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Fig 5.1 — Time domain (frame 0)
    ax = axes[0, 0]
    if fs_val:
        for ch in SENSOR_CHANNELS:
            v = waveforms[ch][0]
            t = np.arange(len(v)) / fs_val * 1000
            ax.plot(t, v*1000, lw=0.5, color=CH_COLORS[ch], label=CH_LABELS[ch], alpha=0.8)
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('U (mV)')
    ax.set_title('Fig. 5.1 — Time Domain, Frame 0')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Fig 5.2 — PSD
    ax2 = axes[0, 1]
    psd_data = {}
    for ch in SENSOR_CHANNELS:
        if waveforms[ch]:
            psds = []
            for v in waveforms[ch]:
                f_w, p = welch(v, fs=fs_val, nperseg=min(512, len(v)//4),
                               window='hann', scaling='density')
                psds.append(p)
            psd_mean = np.mean(psds, axis=0)
            psd_data[ch] = (f_w, psd_mean)
            m = f_w >= 100
            ax2.semilogy(f_w[m]/1e3, psd_mean[m], color=CH_COLORS[ch],
                         lw=0.8, label=CH_LABELS[ch])
    ax2.set_xlabel('Frequency (kHz)'); ax2.set_ylabel('PSD (V²/Hz)')
    ax2.set_title('Fig. 5.2 — Power Spectral Density (Welch, Hann)')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, which='both')

    # Fig 5.3 — Coherence CH1–CH3
    ax3 = axes[1, 0]
    if waveforms[1] and waveforms[3]:
        n_min = min(len(waveforms[1][0]), len(waveforms[3][0]))
        v1 = np.concatenate([v[:n_min] for v in waveforms[1]])
        v3 = np.concatenate([v[:n_min] for v in waveforms[3]])
        f_c, coh13 = coherence(v1, v3, fs=fs_val, nperseg=512)
        m = f_c >= 100
        ax3.plot(f_c[m]/1e3, coh13[m], color='purple', lw=0.8)
        ax3.axhline(0.9, ls='--', color='gray', lw=0.8)
        ax3.set_ylim(0, 1.05)
        ax3.set_xlabel('Frequency (kHz)'); ax3.set_ylabel('$\hat{\gamma}^2_{13}$')
        ax3.set_title('Fig. 5.3 — Coherence CH1 (Soil) – CH3 (Rod A)')
        ax3.grid(True, alpha=0.3)

    # Fig 5.4 — RMS time series
    ax4 = axes[1, 1]
    for ch in SENSOR_CHANNELS:
        rms = [r[f'rms{ch}_V']*1000 for r in records]
        ax4.plot(range(len(rms)), rms, '-', color=CH_COLORS[ch],
                 lw=0.8, label=CH_LABELS[ch], alpha=0.8)
    ax4.set_xlabel('Frame index'); ax4.set_ylabel('RMS (mV)')
    ax4.set_title('Fig. 5.4 — RMS per Frame (all channels)')
    ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'baseline_report.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {OUT_DIR}/baseline_report.png')


## 6  Save Metadata and Export HTML

In [ ]:
meta = {
    'run_id': RUN_ID,
    'notebook': 'notebooks/13_passive_ae_baseline.ipynb',
    'date_utc': datetime.now(timezone.utc).isoformat(),
    'n_frames_requested': N_FRAMES,
    'n_frames_acquired': len(records),
    'channels': SENSOR_CHANNELS,
    'ch_probe': CH_PROBE,
    'ch_scale_V_per_div': CH_SCALE,
    'trigger_source': TRIG_SRC,
    'trigger_level_V': TRIG_LEVEL_V,
    'trigger_slope': 'POS',
    'ch2_status': 'DEFECTIVE — disabled',
    'notes': (
        'Manual probe calibration applied. '
        'No active excitation (passive AE baseline). '
        'CH1 soil piezo installed 2026-06-30, first characterisation run.'
    )
}
with open(OUT_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadata saved: {OUT_DIR}/meta.json')

# HTML export
result = subprocess.run(
    [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'html', '--no-input',
     '--output', 'notebook_report',
     '--output-dir', str(OUT_DIR),
     str(CHAR_ROOT / 'notebooks' / '13_passive_ae_baseline.ipynb')],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'HTML report: {OUT_DIR}/notebook_report.html')
else:
    print('nbconvert error:', result.stderr[-300:])
